In [1]:
import sys, logging, time
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd
from src.feature_extraction import (
    compute_aa_composition, compute_kmer_frequencies, compute_onehot_mean,
    compute_physicochemical, compute_blosum62_profile, compute_length_and_moments,
    compute_esm2
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'

In [2]:
labeled_pd = load_labeled_pd(DATA_DIR)
protein_ids = labeled_pd.index.tolist()
sequences = labeled_pd['sequence'].tolist()
id_to_seq = labeled_pd['sequence'].to_dict()

In [3]:
EXTRACTORS = {
    'aa': compute_aa_composition,
    'kmer2': lambda s: compute_kmer_frequencies(s, k=2),
    'kmer3': lambda s: compute_kmer_frequencies(s, k=3, max_features=500),
    'onehot': compute_onehot_mean,
    'physicochemical': compute_physicochemical,
    'blosum62': compute_blosum62_profile,
    'length_moments': compute_length_and_moments,
    'esm2': lambda s: compute_esm2(s, protein_ids, id_to_seq,
                                    cache_path=DATA_DIR / "esm2_embeddings.parquet")
}
CACHE = {name: DATA_DIR / f"features_{name}.parquet" for name in EXTRACTORS}

In [4]:
feature_dfs = {}
for name, func in EXTRACTORS.items():
    cache_file = CACHE[name]
    if cache_file.exists():
        cached = pd.read_parquet(cache_file)
        if not cached.index.equals(labeled_pd.index):
            need_recompute = True
        else:
            if 'organism' in cached.columns:
                cached = cached.drop(columns=['organism'])
                cached.to_parquet(cache_file)
            df = cached
            log.info(f"Loaded cached {name}: {df.shape}")
            need_recompute = False
    else:
        need_recompute = True

    if need_recompute:
        log.info(f"Computing {name}...")
        t0 = time.time()
        df = func(sequences)
        if len(df) == len(labeled_pd):
            df.index = labeled_pd.index
        else:
            log.warning(f"{name}: got {len(df)} rows, zero-filling.")
            full = pd.DataFrame(0.0, index=labeled_pd.index, columns=df.columns)
            full.iloc[:len(df)] = df.values
            df = full
        df = df.select_dtypes(include=['number'])
        df = df.drop(columns=['organism'], errors='ignore')
        df.to_parquet(cache_file)
        log.info(f"Done in {time.time()-t0:.1f}s, {df.shape}")

    feature_dfs[name] = df

log.info("All features cached (organism‑free).")

2026-06-26 20:48:33,958 INFO Loaded cached aa: (215051, 20)


2026-06-26 20:48:34,192 INFO Loaded cached kmer2: (215051, 400)


2026-06-26 20:48:34,422 INFO Loaded cached kmer3: (215051, 500)


2026-06-26 20:48:34,474 INFO Loaded cached onehot: (215051, 20)


2026-06-26 20:48:34,531 INFO Loaded cached physicochemical: (215051, 7)


...

2026-06-26 21:08:20,647 INFO Saved 215040 embeddings (100.0%).


Batch 611/611... 

2026-06-26 21:08:24,847 INFO Saved 215051 embeddings (100.0%).


2026-06-26 21:08:29,119 INFO Done in 1194.5s, (215051, 320)


2026-06-26 21:08:29,120 INFO All features cached (organism‑free).


In [8]:
# Complete list of 60 organisms
ALL_ORGS = [
    'acidovorax_3H11', 'azobra', 'Btheta', 'Bifido', 'Brev2', 'BFirm',
    'Burk376', 'Burkholderia_OAS925', 'Bvulgatus_CL09T03C04', 'Caulo',
    'CL21', 'Cup4G11', 'PS', 'Dda3937', 'Ddia6719', 'DdiaME23', 'Dino',
    'DvH', 'Dyella79', 'HerbieS', 'Kang', 'Keio', 'Korea', 'Koxy',
    'Lysobacter_OAE881', 'Magneto', 'Marino', 'Methanococcus_JJ',
    'Methanococcus_S2', 'Miya', 'MR1', 'Mucilaginibacter_YX36', 'MycoTube',
    'Pedo557', 'Phaeo', 'Ponti', 'pseudo1_N1B4', 'pseudo13_GW456_L13',
    'pseudo3_N2E3', 'pseudo5_N2C3_1', 'pseudo6_N2E2', 'psRCH2', 'Putida',
    'PV4', 'RalstoniaBSBF1503', 'RalstoniaGMI1000', 'RalstoniaPSI07',
    'RalstoniaUW163', 'rhodanobacter_10B01', 'rhodanobacter_R12',
    'rhodanobacter_T8', 'RPal_CGA009', 'SB2B', 'Smeli', 'SyringaeB728a',
    'SyringaeB728a_mexBdelta', 'SynE', 'Variovorax_OAS795', 'WCS417',
    'Xantho'
]

FB_DIR = DATA_DIR / 'fitness_browser'

In [10]:
from tqdm import tqdm
# ─── Download via CGI endpoints ────────────────────────────────────
BASE_URL = "https://fit.genomics.lbl.gov"
FILES = {
    "experiments": f"{BASE_URL}/cgi-bin/createExpData.cgi?orgId={{org}}",
    "fitness":     f"{BASE_URL}/cgi-bin/createFitData.cgi?orgId={{org}}",
    "proteins":    f"{BASE_URL}/cgi-bin/orgSeqs.cgi?orgId={{org}}",
}

def download_fitness_data(organisms, output_dir, timeout=30):
    """Download required files using CGI endpoints with browser impersonation."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Try curl_cffi first (best bot evasion), fall back to plain requests
    try:
        from curl_cffi.requests import Session
        sess = Session(impersonate="chrome131")
        log.info("Using curl_cffi with Chrome impersonation.")
    except ImportError:
        log.info("curl_cffi not installed; using plain requests with browser headers.")
        sess = requests.Session()
        sess.headers.update({
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })

    # Warm up session
    try:
        sess.get(BASE_URL + "/", timeout=timeout)
    except Exception:
        pass

    for org in tqdm(organisms, desc="Downloading"):
        org_dir = output_dir / org
        org_dir.mkdir(exist_ok=True)
        for label, url_template in FILES.items():
            url = url_template.replace("{org}", org)
            filepath = org_dir / f"{org}_{label}.txt"
            if filepath.exists():
                continue
            try:
                r = sess.get(url, timeout=timeout)
                if r.status_code == 200:
                    filepath.write_bytes(r.content)
            except Exception as e:
                log.warning(f"  {org}/{label}: {e}")

# ─── Download all ───────────────────────────────────────────────────
download_fitness_data(ALL_ORGS, FB_DIR)

2026-06-26 21:10:51,627 INFO Using curl_cffi with Chrome impersonation.


Downloading:   0%|          | 0/60 [00:00<?, ?it/s]

Downloading: 100%|██████████| 60/60 [00:00<00:00, 8267.08it/s]

In [ ]:
# ─── Genome‑level DNA features ────────────────────────────────────
from src.feature_extraction import compute_genome_dna_features

genome_cache = DATA_DIR / 'features_genome_dna.parquet'
if genome_cache.exists():
    genome_feat = pd.read_parquet(genome_cache)
    log.info("Loaded cached genome DNA features.")
else:
    genome_feat = compute_genome_dna_features(FB_DIR, ALL_ORGS, cache_path=genome_cache)
    log.info("Computed genome DNA features and cached.")

# Broadcast per‑organism features to every protein row
org_col = labeled_pd['organism']
genome_feat_aligned = genome_feat.reindex(org_col, fill_value=0)
genome_feat_aligned.index = labeled_pd.index

# Save as a standard feature parquet — no feature_dfs needed
genome_feat_aligned.to_parquet(DATA_DIR / 'features_genome_dna.parquet')
log.info(f"Genome DNA features saved: {genome_feat_aligned.shape}")